# 株価データ取得プログラム
このノートブックは、指定した銘柄の株価データを取得し、CSVファイルとしてダウンロードします。

## 1. 必要なライブラリのインストールと読み込み

In [ ]:
# 必要なライブラリをインストール
!pip install yfinance pandas

# ライブラリの読み込み
import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta
from google.colab import files

## 2. 株価データ取得の設定
ここで銘柄コード、期間を設定します

In [ ]:
# ===== ここを編集してください =====

# 銘柄コード（例）
# 日本株: "7203.T" (トヨタ自動車)、"6758.T" (ソニー)
# 米国株: "AAPL" (Apple)、"MSFT" (Microsoft)、"GOOGL" (Google)
ticker_symbol = "7203.T"  # ここに取得したい銘柄コードを入力

# 期間設定（開始日と終了日）
start_date = "2024-01-01"  # 開始日 (YYYY-MM-DD形式)
end_date = "2025-01-24"    # 終了日 (YYYY-MM-DD形式)

# データの間隔（"1d"=日次、"1wk"=週次、"1mo"=月次）
interval = "1d"

# ===== 入力値の検証 =====
import re
from datetime import datetime

# 日付形式の検証
date_pattern = r'^\d{4}-\d{2}-\d{2}$'
if not re.match(date_pattern, start_date) or not re.match(date_pattern, end_date):
    raise ValueError("日付は YYYY-MM-DD 形式で入力してください")

# 日付の妥当性チェック
try:
    start_dt = datetime.strptime(start_date, "%Y-%m-%d")
    end_dt = datetime.strptime(end_date, "%Y-%m-%d")
    if start_dt >= end_dt:
        raise ValueError("開始日は終了日より前の日付にしてください")
except ValueError as e:
    raise ValueError(f"日付が無効です: {e}")

# 銘柄コードの検証
if not ticker_symbol or ticker_symbol.strip() == "":
    raise ValueError("銘柄コードを入力してください")

# インターバルの検証
valid_intervals = ["1m", "2m", "5m", "15m", "30m", "60m", "90m", "1h", "1d", "5d", "1wk", "1mo", "3mo"]
if interval not in valid_intervals:
    raise ValueError(f"データ間隔は次のいずれかを指定してください: {', '.join(valid_intervals)}")

print(f"銘柄コード: {ticker_symbol}")
print(f"期間: {start_date} から {end_date}")
print(f"データ間隔: {interval}")
print("✓ 入力値の検証が完了しました")

## 3. 株価データの取得

In [ ]:
# 株価データを取得
print(f"\n{ticker_symbol} のデータを取得中...")

# yfinanceを使用してデータ取得（エラーハンドリング追加）
try:
    stock_data = yf.download(
        ticker_symbol,
        start=start_date,
        end=end_date,
        interval=interval,
        progress=True
    )
except Exception as e:
    error_msg = f"❌ データ取得中にエラーが発生しました: {str(e)}\n"
    error_msg += "以下を確認してください:\n"
    error_msg += "  - 銘柄コードが正しいか\n"
    error_msg += "  - インターネット接続が正常か\n"
    error_msg += "  - 日付範囲が適切か"
    raise RuntimeError(error_msg)

# データが取得できたか確認（重要: 空の場合は処理を中断）
if stock_data.empty:
    error_msg = "⚠️ データが取得できませんでした。\n"
    error_msg += "以下を確認してください:\n"
    error_msg += f"  - 銘柄コード「{ticker_symbol}」が正しいか\n"
    error_msg += f"  - 期間「{start_date}」から「{end_date}」にデータが存在するか\n"
    error_msg += "  - 日本株の場合は末尾に「.T」が付いているか (例: 7203.T)\n"
    error_msg += "\n⚠️ これ以降のセルは実行しないでください（空のファイルがダウンロードされます）"
    raise ValueError(error_msg)

# データ取得成功
print(f"✓ データ取得完了: {len(stock_data)} 件のデータ")
print(f"✓ カラム数: {len(stock_data.columns)}")
print(f"✓ カラム名: {', '.join(map(str, stock_data.columns))}")
print("\n最初の5件:")
print(stock_data.head())
print("\n最後の5件:")
print(stock_data.tail())

## 4. データの整形と確認

In [ ]:
# データフレームの情報を表示
print("データの概要:")
print(stock_data.info())

print("\n基本統計量:")
print(stock_data.describe())

# カラム名を日本語に変更（オプション）
# 注意: yfinanceのカラム構造が変更された場合に対応
stock_data_jp = stock_data.copy()

# 日本語化フラグの初期化
columns_converted_to_japanese = False

# カラム数の検証
expected_columns = 6
if len(stock_data_jp.columns) != expected_columns:
    print(f"⚠️ 警告: 予期しないカラム数です（期待: {expected_columns}, 実際: {len(stock_data_jp.columns)}）")
    print(f"カラム名: {list(map(str, stock_data_jp.columns))}")
    print("日本語カラム名への変換をスキップします。")
else:
    # 期待されるカラム名を検証（文字列に変換して比較）
    expected_cols = ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']
    actual_cols = [str(col) for col in stock_data_jp.columns]
    
    if actual_cols == expected_cols:
        stock_data_jp.columns = ['始値', '高値', '安値', '終値', '調整後終値', '出来高']
        columns_converted_to_japanese = True  # フラグを設定
        print("✓ カラム名を日本語に変換しました")
        print("\n日本語カラム名でのプレビュー:")
        print(stock_data_jp.head())
    else:
        print(f"⚠️ 警告: カラム名が期待と異なります")
        print(f"期待: {expected_cols}")
        print(f"実際: {actual_cols}")
        print("日本語カラム名への変換をスキップします。元のカラム名を使用してください。")

## 5. CSVファイルとしてダウンロード

In [ ]:
# ファイル名を生成
filename = f"{ticker_symbol}_{start_date}_to_{end_date}.csv"

# データの再確認（念のため）
if stock_data.empty:
    raise ValueError("❌ データが空です。このセルを実行する前にデータを取得してください。")

# CSVファイルとして保存（元のカラム名）
try:
    stock_data.to_csv(filename, encoding='utf-8-sig')  # utf-8-sigでExcelでも文字化けしない
    print(f"✓ CSVファイルを作成しました: {filename}")
    print(f"  - 行数: {len(stock_data)}")
    print(f"  - カラム数: {len(stock_data.columns)}")
except Exception as e:
    raise RuntimeError(f"❌ CSVファイルの作成に失敗しました: {str(e)}")

# ファイルをダウンロード
try:
    print("\nダウンロードを開始します...")
    files.download(filename)
    print("✓ ダウンロード完了！")
except Exception as e:
    print(f"⚠️ ダウンロードに失敗しました: {str(e)}")
    print(f"ファイル「{filename}」は作成されていますが、ダウンロードできませんでした。")

## 6. (オプション) 日本語カラム名でもダウンロード

In [ ]:
# 日本語カラム名バージョンもダウンロードしたい場合

# stock_data_jpが存在するか確認
try:
    if stock_data_jp.empty:
        raise ValueError("データが空です")
except NameError:
    raise NameError("❌ 「stock_data_jp」が定義されていません。先にセル8を実行してください。")

# 日本語化フラグの確認
try:
    is_japanese = columns_converted_to_japanese
except NameError:
    # フラグが未定義の場合、カラム名から判定
    is_japanese = '始値' in [str(col) for col in stock_data_jp.columns]

# ファイル名を動的に決定
if is_japanese:
    filename_jp = f"{ticker_symbol}_{start_date}_to_{end_date}_日本語.csv"
else:
    filename_jp = f"{ticker_symbol}_{start_date}_to_{end_date}_copy.csv"
    print("⚠️ カラムが日本語化されていないため、ファイル名を「_copy.csv」としました")

# CSVファイルとして保存
try:
    stock_data_jp.to_csv(filename_jp, encoding='utf-8-sig')
    if is_japanese:
        print(f"✓ 日本語版CSVファイルを作成しました: {filename_jp}")
    else:
        print(f"✓ CSVファイルを作成しました: {filename_jp}")
    print(f"  - 行数: {len(stock_data_jp)}")
    print(f"  - カラム数: {len(stock_data_jp.columns)}")
    print(f"  - カラム名: {', '.join(map(str, stock_data_jp.columns))}")
except Exception as e:
    raise RuntimeError(f"❌ CSVファイルの作成に失敗しました: {str(e)}")

# ファイルをダウンロード
try:
    files.download(filename_jp)
    print("✓ ダウンロード完了！")
except Exception as e:
    print(f"⚠️ ダウンロードに失敗しました: {str(e)}")
    print(f"ファイル「{filename_jp}」は作成されていますが、ダウンロードできませんでした。")

---
## 使い方のヒント

### 主要な日本株の銘柄コード例:
- トヨタ自動車: `7203.T`
- ソニーグループ: `6758.T`
- ソフトバンクグループ: `9984.T`
- キーエンス: `6861.T`
- 三菱UFJフィナンシャル・グループ: `8306.T`

### 主要な米国株の銘柄コード例:
- Apple: `AAPL`
- Microsoft: `MSFT`
- Google (Alphabet): `GOOGL`
- Amazon: `AMZN`
- Tesla: `TSLA`

### データ間隔の設定:
- `1d`: 日次データ
- `1wk`: 週次データ
- `1mo`: 月次データ
- `1h`: 時間足（直近60日間のみ）

### カラムの説明:
- **Open (始値)**: その期間の最初の取引価格
- **High (高値)**: その期間の最高価格
- **Low (安値)**: その期間の最低価格
- **Close (終値)**: その期間の最後の取引価格
- **Adj Close (調整後終値)**: 株式分割や配当を考慮した終値
- **Volume (出来高)**: その期間の取引量